# 🚗 Drowsy Driver — PIPELINE MASTER (Colab T4)

Pipeline end-to-end: **EDA → Tiền xử lý/Augmentation → Train 4 mô hình → Đánh giá & So sánh**.

| Nhánh | Mô hình | Output |
|---|---|---|
| M1 | EAR/MAR + MediaPipe (baseline rule-based) | ngưỡng EAR/MAR |
| M2 | CNN + MediaPipe (eye 64×64) | `cnn_eye.tflite` |
| M3 | YOLOv11s | `yolo11s_best.pt` |
| M4 | YOLO26m | `yolo26m_best.pt` |

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`, điền `ROBOFLOW_API_KEY` ở Cell 2.

Tham số tốt nhất đồng bộ với `data_pipeline/best_params.json` và `configs/yolo_hparams.yaml` trong repo.


In [ ]:
#@title Cell 1 — Kiểm tra GPU & cài thư viện
!nvidia-smi
%pip install -q "ultralytics>=8.4.0" roboflow mediapipe opencv-python-headless matplotlib pandas
import tensorflow as tf, sys
print("TF:", tf.__version__, "| Python:", sys.version.split()[0])
print("GPU TF:", tf.config.list_physical_devices('GPU'))

In [ ]:
#@title Cell 2 — Tải dataset Roboflow (YOLO 11 lớp) + MRL eye (classification)
import os
HOME = os.getcwd()
ROBOFLOW_API_KEY = ""  #@param {type:"string"}

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# Dataset 11 lớp khớp summary_yolo26.json (faseeh-f2cpp/drowsiness-driver-yqss9)
project = rf.workspace("faseeh-f2cpp").project("drowsiness-driver-yqss9")
ds = project.version(1).download("yolov11", location=f"{HOME}/ds_yolo")
DATA_YAML = f"{ds.location}/data.yaml"
print("DATA_YAML:", DATA_YAML)

# (Tuỳ chọn) MRL eye dataset cho CNN — upload kaggle.json rồi bỏ comment:
# !pip -q install kaggle && mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d prasadvpatil/mrl-dataset -p {HOME}/mrl --unzip
CNN_DATA = f"{HOME}/mrl"  # cấu trúc kỳ vọng: train|val|test / eyes_closed|eyes_open

## 📊 Bước 1 — EDA
Thống kê phân phối lớp, kích thước bbox, ảnh mẫu. Kết quả lưu `outputs_eda/`.

In [ ]:
#@title Cell 3 — EDA dataset YOLO
import yaml, glob, collections, os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

os.makedirs("outputs_eda", exist_ok=True)
with open(DATA_YAML) as f: cfg = yaml.safe_load(f)
CLASSES = cfg["names"]; print(len(CLASSES), "classes:", CLASSES)

stats, box_sizes = collections.Counter(), []
for split in ["train", "valid", "test"]:
    for lb in glob.glob(f"{ds.location}/{split}/labels/*.txt"):
        for line in open(lb):
            p = line.split()
            if len(p) >= 5:
                stats[(split, CLASSES[int(p[0])])] += 1
                box_sizes.append((float(p[3]), float(p[4])))

df = pd.DataFrame([{"split": s, "class": c, "n_boxes": n} for (s, c), n in stats.items()])
pv = df.pivot_table(index="class", columns="split", values="n_boxes", fill_value=0)
print(pv)
pv.to_csv("outputs_eda/class_distribution.csv")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
pv.plot(kind="barh", stacked=True, ax=ax[0], title="Phân phối bbox theo lớp/split")
bw, bh = zip(*box_sizes)
ax[1].hist2d(bw, bh, bins=40); ax[1].set_title("Kích thước bbox tương đối (w,h)")
plt.tight_layout(); plt.savefig("outputs_eda/eda_yolo.png", dpi=120); plt.show()

# Nhận xét tự động về lớp trùng ngữ nghĩa
dup = [c for c in CLASSES if c.lower() in ("close", "closed", "eyeclosed", "yawn")]
print("⚠️ Lớp trùng ngữ nghĩa cần cân nhắc gộp:", dup)

In [ ]:
#@title Cell 4 — Lưới ảnh mẫu
import cv2, random
imgs = glob.glob(f"{ds.location}/train/images/*")[:: max(1, len(glob.glob(f"{ds.location}/train/images/*")) // 9)][:9]
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, p in zip(axes.flat, imgs):
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.suptitle("Ảnh mẫu train"); plt.savefig("outputs_eda/sample_grid.png", dpi=120); plt.show()

## 🔧 Bước 2 — Tiền xử lý & Augmentation
Minh hoạ trước/sau từng phép biến đổi (resize, flip, rotate, noise, contrast, letterbox+stride).
Với YOLO, augmentation (HSV, mosaic, fliplr...) do ultralytics thực hiện on-the-fly theo tham số ở Cell train.

In [ ]:
#@title Cell 5 — Demo augmentation trước/sau
import cv2, numpy as np, matplotlib.pyplot as plt

img = cv2.cvtColor(cv2.imread(imgs[0]), cv2.COLOR_BGR2RGB)

def letterbox(im, size=640, stride=32):
    h, w = im.shape[:2]; r = size / max(h, w)
    nh, nw = int(h * r), int(w * r)
    nh, nw = nh - nh % stride or stride, nw - nw % stride or stride
    return cv2.resize(im, (nw, nh))

augs = {
    "Gốc": img,
    "Resize 64×64 (CNN)": cv2.resize(img, (64, 64)),
    "Letterbox 640 + stride 32 (YOLO)": letterbox(img),
    "Flip ngang": cv2.flip(img, 1),
    "Xoay +15°": cv2.warpAffine(img, cv2.getRotationMatrix2D((img.shape[1]//2, img.shape[0]//2), 15, 1), img.shape[1::-1]),
    "Gaussian noise σ=0.03": np.clip(img/255. + np.random.normal(0, .03, img.shape), 0, 1),
    "Tăng contrast +20%": np.clip(img/255. * 1.2, 0, 1),
    "Giảm sáng (đêm)": np.clip(img/255. * .5, 0, 1),
}
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (name, im) in zip(axes.flat, augs.items()):
    ax.imshow(im); ax.set_title(name, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.savefig("outputs_eda/augmentation_demo.png", dpi=120); plt.show()

## 🟢 Bước 3 — M1: Baseline EAR/MAR + MediaPipe
Không cần train. Tính EAR/MAR từ 478 landmark; ngưỡng tốt nhất (đã tinh chỉnh): **EAR<0.21 ≥1.5s ⇒ DROWSY**, **MAR>0.6 ≥1.0s ⇒ YAWNING**.

In [ ]:
#@title Cell 6 — Hàm EAR/MAR + chạy thử trên ảnh mẫu
import mediapipe as mp
import numpy as np, cv2

mp_face = mp.solutions.face_mesh
LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH     = [61, 81, 311, 291, 178, 402]

def aspect_ratio(pts):
    a = np.linalg.norm(pts[1] - pts[5]); b = np.linalg.norm(pts[2] - pts[4])
    c = np.linalg.norm(pts[0] - pts[3]);  return (a + b) / (2 * c + 1e-6)

EAR_T, MAR_T = 0.21, 0.60  # best params (xem docs/PIPELINE_E2E_TRACKING.md)

with mp_face.FaceMesh(static_image_mode=True, refine_landmarks=True) as fm:
    for p in imgs[:3]:
        im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        res = fm.process(im)
        if not res.multi_face_landmarks:
            print(p.split('/')[-1], "→ không thấy mặt"); continue
        lm = res.multi_face_landmarks[0].landmark
        h, w = im.shape[:2]
        xy = lambda idx: np.array([[lm[i].x * w, lm[i].y * h] for i in idx])
        ear = (aspect_ratio(xy(LEFT_EYE)) + aspect_ratio(xy(RIGHT_EYE))) / 2
        mar = aspect_ratio(xy(MOUTH))
        state = "DROWSY?" if ear < EAR_T else ("YAWN?" if mar > MAR_T else "AWAKE")
        print(f"{p.split('/')[-1]:40s} EAR={ear:.3f} MAR={mar:.3f} → {state}")

## 🔵 Bước 4 — M2: CNN + MediaPipe (eye classifier)
Siêu tham số tốt nhất từ `best_params.json` (test acc **0.9866**). Cần dataset classification (MRL) ở Cell 2.

In [ ]:
#@title Cell 7 — Train CNN eye 64×64 (best params)
import tensorflow as tf, os, json
from tensorflow.keras import layers as L

BEST = dict(image_size=64, batch_size=32, epochs=25, lr=1e-3, dropout=0.3)

if os.path.isdir(f"{CNN_DATA}/train"):
    mk = lambda s, sh: tf.keras.utils.image_dataset_from_directory(
        f"{CNN_DATA}/{s}", image_size=(64, 64), batch_size=BEST["batch_size"],
        label_mode="categorical", shuffle=sh)
    train_ds, val_ds, test_ds = mk("train", True), mk("val", False), mk("test", False)
    norm = lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)   # KHÔNG Rescaling trong model (tránh normalize 2 lần trên Android)
    train_ds, val_ds, test_ds = (d.map(norm).prefetch(tf.data.AUTOTUNE) for d in (train_ds, val_ds, test_ds))

    aug = tf.keras.Sequential([L.RandomFlip("horizontal"), L.RandomRotation(0.05),
                               L.RandomZoom(0.10), L.RandomContrast(0.20)])
    model = tf.keras.Sequential([
        L.Input((64, 64, 3)), aug,
        L.Conv2D(32, 3, activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(64, 3, activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(128, 3, activation="relu"), L.BatchNormalization(), L.GlobalAveragePooling2D(),
        L.Dropout(BEST["dropout"]), L.Dense(2, activation="softmax")])
    model.compile(tf.keras.optimizers.Adam(BEST["lr"]), "categorical_crossentropy", ["accuracy"])
    hist = model.fit(train_ds, validation_data=val_ds, epochs=BEST["epochs"],
        callbacks=[tf.keras.callbacks.EarlyStopping("val_loss", patience=4, restore_best_weights=True)])
    test_loss, test_acc = model.evaluate(test_ds)
    print(f"✅ CNN eye test accuracy = {test_acc:.4f}  (best repo run: 0.9866)")

    # Export TFLite cho Android/iOS
    tfl = tf.lite.TFLiteConverter.from_keras_model(model).convert()
    open("cnn_eye.tflite", "wb").write(tfl)
    json.dump({"model": "DrowsyCNN_Eye", "test_accuracy": float(test_acc), **BEST},
              open("summary_cnn_eye.json", "w"), indent=2)
else:
    test_acc = None
    print("⚠️ Chưa có CNN_DATA — bỏ qua (dùng kết quả repo: test acc 0.9866)")

## 🟡 Bước 5 — M3: YOLOv11s
Tham số đồng bộ `configs/yolo_hparams.yaml` — augment nhẹ tay cho khuôn mặt (degrees 5, flipud 0, mosaic 0.5, mixup 0).

In [ ]:
#@title Cell 8 — Train YOLOv11s
from ultralytics import YOLO

YOLO_HP = dict(epochs=50, imgsz=640, batch=-1, patience=15, cos_lr=True,
               lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=5e-4, warmup_epochs=3,
               box=7.5, cls=0.5, dfl=1.5,
               hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, degrees=5.0, translate=0.1, scale=0.5,
               fliplr=0.5, flipud=0.0, mosaic=0.5, mixup=0.0, close_mosaic=10)

m3 = YOLO("yolo11s.pt")
r3 = m3.train(data=DATA_YAML, name="drowsy_yolo11s", **YOLO_HP)
v3 = m3.val()
map50_y11 = float(v3.box.map50)
print(f"✅ YOLOv11s mAP50 = {map50_y11:.4f} | mAP50-95 = {float(v3.box.map):.4f}")

## 🟠 Bước 6 — M4: YOLO26m
Kiến trúc NMS-free tối ưu edge. Run trước đó: 10 epoch → mAP50 0.502; lần này 50 epoch.

In [ ]:
#@title Cell 9 — Train YOLO26m
m4 = YOLO("yolo26m.pt")
r4 = m4.train(data=DATA_YAML, name="drowsy_yolo26", **YOLO_HP)
v4 = m4.val()
map50_y26 = float(v4.box.map50)
print(f"✅ YOLO26m mAP50 = {map50_y26:.4f} | mAP50-95 = {float(v4.box.map):.4f}")
import json
json.dump({"model": "yolo26m", "epochs": YOLO_HP["epochs"], "map50": map50_y26},
          open("summary_yolo26_new.json", "w"), indent=2)

## 🏁 Bước 7 — Tổng hợp, so sánh & kết luận

In [ ]:
#@title Cell 10 — Bảng so sánh 4 mô hình
import pandas as pd

rows = [
    ["M1 EAR/MAR + MediaPipe", "rule-based", "acc kịch bản", "~0.90", "25-30 FPS", "0 KB", "cao"],
    ["M2 CNN eye + MediaPipe", "classification", "test acc",
     f"{test_acc:.4f}" if test_acc else "0.9866 (repo)", "20-25 FPS", "46 KB", "trung bình"],
    ["M3 YOLOv11s", "detection 11 lớp", "mAP50",
     f"{map50_y11:.4f}" if 'map50_y11' in dir() else "—", "8-12 FPS", "~19 MB", "thấp"],
    ["M4 YOLO26m", "detection 11 lớp", "mAP50",
     f"{map50_y26:.4f}" if 'map50_y26' in dir() else "0.502 (10ep)", "10-15 FPS", "~40 MB", "thấp"],
]
cmp = pd.DataFrame(rows, columns=["Mô hình", "Loại", "Metric", "Kết quả", "FPS (phone CPU)", "Size", "Giải thích được"])
cmp.to_csv("model_comparison.csv", index=False)
cmp

### 📌 Nhận xét tổng quát

1. **MediaPipe + CNN là lựa chọn production** cho mobile: acc ~98.7%, model 46 KB, ≥20 FPS CPU, có EAR/MAR giải thích được.
2. **EAR/MAR baseline** tốt ở kịch bản rõ ràng, suy giảm khi thiếu sáng/đeo kính → CNN bù đắp.
3. **YOLO end-to-end** mạnh về phát hiện trên full-frame nhưng mAP bị kéo xuống do 11 lớp trùng ngữ nghĩa (`close/closed/Eyeclosed`) → gộp lớp xuống 5–6 rồi so sánh lại.
4. **Hướng temporal** (Transformer chuỗi 20 EAR — test acc 0.9988) giảm báo nhầm do chớp mắt, là điểm khác biệt chính của đề tài.

**Tải kết quả:** `model_comparison.csv`, `cnn_eye.tflite`, `runs/detect/*/weights/best.pt`, `outputs_eda/*` — cập nhật vào `docs/PIPELINE_E2E_TRACKING.md` (bảng Experiment Log).